# Play Obstacle Environments

Choose an environment with the buttons, preview an `rgb_array` frame inline, then run the last cell to play in a pygame window.

In [ ]:
import ipywidgets as widgets
import numpy as np
from IPython.display import display
from PIL import Image
from masa.common.notebook_play import make_reset_env, sync_selected_env

VALID_ENV_NAMES = ("Obstacle", "ObstacleV2", "ObstacleV3", "ObstacleV4")

def make_env(env_name, **kwargs):
    if env_name == "Obstacle":
        from masa.envs.continuous.obstacle import Obstacle

        return Obstacle(**kwargs)
    if env_name == "ObstacleV2":
        from masa.envs.continuous.obstacle_v2 import ObstacleV2

        return ObstacleV2(**kwargs)
    if env_name == "ObstacleV3":
        from masa.envs.continuous.obstacle_v3 import ObstacleV3

        return ObstacleV3(**kwargs)
    if env_name == "ObstacleV4":
        from masa.envs.continuous.obstacle_v4 import ObstacleV4

        return ObstacleV4(**kwargs)
    raise ValueError(f"env_name must be one of {VALID_ENV_NAMES!r}")

ENV_SELECTOR = widgets.ToggleButtons(
    options=VALID_ENV_NAMES,
    value="ObstacleV2",
    description="Env",
)
display(ENV_SELECTOR)

ENV_NAME = ENV_SELECTOR.value
SEED = 0


In [ ]:
ENV_NAME = ENV_SELECTOR.value
preview_env = make_env(ENV_NAME, render_mode="rgb_array", render_window_size=512)
obs, info = preview_env.reset(seed=SEED)
print("reset:", {"obs": obs, "info": info})
display(Image.fromarray(preview_env.render()))
preview_env.close()


In [ ]:
def action_from_pressed(pressed, pygame):
    if pressed[pygame.K_SPACE]:
        return np.array([0.0, 0.0], dtype=np.float32)
    action = np.array([0.0, 0.0], dtype=np.float32)
    if pressed[pygame.K_LEFT] or pressed[pygame.K_a]:
        action[0] -= 1.0
    if pressed[pygame.K_RIGHT] or pressed[pygame.K_d]:
        action[0] += 1.0
    if pressed[pygame.K_UP] or pressed[pygame.K_w]:
        action[1] += 1.0
    if pressed[pygame.K_DOWN] or pressed[pygame.K_s]:
        action[1] -= 1.0
    if not np.any(action):
        return None
    return action


def _print_reset(obs, info):
    print("reset:", {"obs": obs, "info": info})


def play_env(env_name=None, seed=SEED):
    import pygame

    follow_selector = env_name is None
    if follow_selector:
        env_name = ENV_SELECTOR.value
    env, obs, info = make_reset_env(make_env, env_name, seed=seed, render_mode="human", render_window_size=512)
    finished = False
    print("Controls: hold arrows or WASD to accelerate, Space neutral, R resets, Q or Escape quits.")
    _print_reset(obs, info)

    try:
        running = True
        while running and not env.human_window_closed:
            if follow_selector:
                env, env_name, obs, info, switched = sync_selected_env(
                    env,
                    env_name,
                    ENV_SELECTOR,
                    make_env,
                    seed=seed,
                    render_window_size=512,
                    pygame=pygame,
                )
                if switched:
                    finished = False
                    print("switched:", env_name)
                    _print_reset(obs, info)

            for event in pygame.event.get():
                if not env.handle_pygame_event(event):
                    running = False
                    break
                if event.type != pygame.KEYDOWN:
                    continue
                if event.key in (pygame.K_q, pygame.K_ESCAPE):
                    running = False
                    break
                if event.key == pygame.K_r:
                    obs, info = env.reset(seed=seed)
                    finished = False
                    _print_reset(obs, info)
                    continue

            if not running:
                break

            action = None if finished else action_from_pressed(pygame.key.get_pressed(), pygame)

            if action is None:
                env.render()
                continue

            obs, reward, terminated, truncated, info = env.step(action)
            finished = terminated or truncated
            print({"obs": obs, "reward": reward, "terminated": terminated, "truncated": truncated, "info": info})
    finally:
        env.close()

play_env(seed=SEED)
